# Flood-Paper RAG — Interactive Query Notebook

Use this notebook to explore the vector store, run custom queries, and inspect extracted results.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # project root

from src.utils.logging_config import configure_logging
configure_logging(level='INFO')


## 1  Initialise the pipeline (reads existing ChromaDB — no re-ingestion)

In [2]:
from src.pipeline.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(extractor_type='regex')   # change to 'ollama' if Ollama is running


/home/niko_notebook/projects/paper_satelit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-04-27 10:13:01  INFO      src.pipeline.rag_pipeline  Initialising components …
2026-04-27 10:13:01  INFO      src.embedding.embedder  Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 (device=cpu)


2026-04-27 10:13:01  WARNING   huggingface_hub.utils._http  Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3204.89it/s]


2026-04-27 10:13:04  INFO      src.embedding.embedder  Embedding dimension: 384
2026-04-27 10:13:04  INFO      src.vectorstore.chroma_store  VectorStore ready: collection=flood_papers  existing_docs=13970
2026-04-27 10:13:04  INFO      src.pipeline.rag_pipeline  Extractor: RegexExtractor


/home/niko_notebook/projects/paper_satelit/src/embedding/embedder.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


## 2  Check vector store status

In [ ]:
from src.vectorstore.chroma_store import VectorStore
store = VectorStore()
print(f"Chunks in store: {store.count()}")


## 3  Run a custom semantic query

In [ ]:
from src.embedding.embedder import Embedder

embedder = Embedder()
query = "U-Net flood segmentation Sentinel-1 F1 score IoU"
vec   = embedder.embed_query(query)
hits  = store.query(vec, top_k=5)

for i, h in enumerate(hits, 1):
    print(f"--- Hit {i}  [{h['filename']}  p{h['page_start']}]  dist={h['distance']:.4f}")
    print(h['text'][:300])
    print()


## 4  Extract structured data from all indexed papers

In [ ]:
import pandas as pd

df = pipeline.query(save_csv=True)
print(df.shape)
df.head(20)


## 5  Inspect accuracy distributions

In [ ]:
import matplotlib.pyplot as plt

acc = df.dropna(subset=['OA', 'F1', 'IoU'], how='all').copy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, ['OA', 'F1', 'IoU']):
    vals = acc[col].dropna()
    if len(vals):
        ax.hist(vals, bins=10, edgecolor='black')
    ax.set_title(col)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Value (0-1 scale)')
    ax.set_ylabel('Count')

plt.suptitle('Accuracy metric distributions across indexed papers')
plt.tight_layout()
plt.savefig('../outputs/metric_distributions.png', dpi=150)
plt.show()


## 6  Accuracy level breakdown

In [ ]:
counts = df['Accuracy_Level'].value_counts()
print(counts.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, edgecolor='black')
ax.set_ylabel('Papers')
ax.set_title('Accuracy Level Classification')
plt.tight_layout()
plt.savefig('../outputs/accuracy_levels.png', dpi=150)
plt.show()


## 7  Filter and export custom subsets

In [ ]:
# papers with at least one numeric metric
quantitative = df[df['Accuracy_Level'].isin(['Quantitative', 'Semi-quantitative'])].copy()
print(f"{len(quantitative)} papers with numeric metrics")

# sort by F1 then OA
quantitative_sorted = quantitative.sort_values(['F1', 'OA'], ascending=False)
quantitative_sorted[['Author','Method','Sensor','OA','F1','IoU','Accuracy_Level','Source_File']].head(20)


## 8  Query a single file

In [ ]:
# replace with an actual filename from your data/literature/ folder
# filename = 'smith_2024_flood.pdf'
# result = pipeline.query_file(filename)
# print(result)
print("Set filename above and uncomment to query a single paper.")
